In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
len(documents)

72

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
data_gen_instructions

"You emulate a student who is taking our LLM course.\nYou are given one lesson page from the course.\nFormulate 5 questions this student might ask that are answered by this page.\n\nRules:\n- The page should contain the answer to each question.\n- Make the questions complete and not too short.\n- Use as few words as possible from the page; don't copy its phrasing.\n- The questions should resemble how people actually ask things online:\n  not too formal, not too short, not too long.\n- Ask about the content of the lesson, not about its formatting or filename."

In [11]:
documents[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [13]:
pages = [
    '01-agentic-rag/lessons/01-intro.md',
    '01-agentic-rag/lessons/02-environment.md',
    '01-agentic-rag/lessons/03-rag.md'
]
pages

['01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/02-environment.md',
 '01-agentic-rag/lessons/03-rag.md']

In [15]:
docs=[]
for d in documents:
    if d['filename'] in pages:
        docs.append(d)

In [16]:
len(docs)

3

In [32]:
for i in docs:
    print(i['filename'])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [19]:
## Q1. What's the average number of input tokens across these 3 calls?

In [20]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [21]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [29]:
import json

In [36]:
input_tokens_sum = 0
for i in docs:
    user_prompt = json.dumps(i)
    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ]
    
    response = openai_client.responses.parse(
        model="gpt-5.4-mini",
        input=messages,
        text_format=Questions
    )

    input_tokens = response.usage.input_tokens
    print(input_tokens)
    input_tokens_sum += input_tokens

print()
print(f'Avg input tokens: {input_tokens_sum/3.0}')

1020
1286
1753

Avg input tokens: 1353.0


In [37]:
### 1353
### approx 1400

In [38]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [45]:
from tqdm.auto import tqdm
import numpy as np
from embedder import Embedder
embed = Embedder()

X = []

for i in tqdm(range(0, len(chunks))):
    batch_vectors = embed.encode_batch([chunks[i]['content']])
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/295 [00:00<?, ?it/s]

In [84]:
## Q2. After running text_search for it, what's the filename of the first result?

In [85]:
from minsearch import VectorSearch, Index

In [86]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [87]:
def text_search(query, num_results=10):
    tindex = Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    )
    tindex.fit(chunks)
    return tindex.search(query=query, num_results=num_results, filter_dict={})


def vector_search(query, num_results=10):
    vindex = VectorSearch(keyword_fields=["filename"])
    v = embed.encode(query)
    vindex.fit(X, chunks)
    return vindex.search(v, {}, num_results)


def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

    

In [88]:
import pandas as pd
ground_truth_df = pd.read_csv('ground-truth.csv')
ground_truth_df.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [89]:
len(ground_truth_df)

360

In [90]:
ground_truth =  ground_truth_df.to_dict(orient='records')
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [91]:
results = text_search(q)
results[0]['filename']

'01-agentic-rag/lessons/03-rag.md'

In [92]:
### 01-agentic-rag/lessons/03-rag.md

In [93]:
## Q3. After running vector_search for the same question, what's the filename of the first result?

In [94]:
results = vector_search(q)
results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [95]:
### 01-agentic-rag/lessons/01-intro.md

In [96]:
## Q4. Evaluate text_search on the ground truth data. What's the Hit Rate?

In [97]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [98]:
relevance = compute_relevance_total(ground_truth, text_search)
relevance

  0%|          | 0/360 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 1, 0, 1, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 1, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
 [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 1, 0, 1, 0, 0, 0, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 1, 1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
 [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
 [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
 [1, 0, 0,

In [99]:
hit_rate(relevance)

0.8416666666666667

In [100]:
### 0.88

In [101]:
## Q5. Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search. What's the MRR?

In [102]:
relevance = compute_relevance_total(ground_truth, vector_search)
relevance

  0%|          | 0/360 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0, 0, 1, 0, 0, 0],
 [0, 0, 1, 1, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0, 1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 1, 1, 1, 0, 0, 1, 0, 0],
 [1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
 [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 1, 0, 0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0,

In [103]:
mrr(relevance)

0.5646472663139328

In [104]:
### 0.55

In [105]:
## Q6. Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. 
## Compare the MRR values for these runs. Which k gives the best MRR?

In [106]:
def compute_relevance(q, search_function, k):
    doc_id = q["filename"]
    results = search_function(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function, k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, k=k)
        relevance_total.append(relevance)

    return relevance_total

In [107]:
results = []

In [108]:
for k in [1, 50, 100, 200]:
    relevance = compute_relevance_total(ground_truth, hybrid_search, k)
    results.append({
        'k': k,
        'mrr': mrr(relevance)
    })
    print(f'k={k}, mrr={mrr(relevance)}')

  0%|          | 0/360 [00:00<?, ?it/s]

k=1, mrr=0.6481944444444449


  0%|          | 0/360 [00:00<?, ?it/s]

k=50, mrr=0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

k=100, mrr=0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

k=200, mrr=0.637916666666667


In [109]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(5)

,k,mrr
0,1,0.648194
1,50,0.637917
2,100,0.637917
3,200,0.637917


In [110]:
### k=1